# The critic as another model, and progress under a prover

`07`'s critic was a linear head on the policy's own hidden states (layer 23 of Qwen3.5-0.8B). Its PPO run from the recovery seed was the series' first to lift a greedy evaluation above step 0 (0.586 → 0.658 at step 40, 60 dev questions). This notebook keeps 07's seed, task, reward and PPO configuration and replaces the critic with **another model**, following

> Setlur, Nagpal, Fisch, Geng, Eisenstein, Agarwal, Agarwal, Berant, Kumar. *Rewarding Progress: Scaling Automated Process Verifiers for LLM Reasoning.* Google Research / Google DeepMind / CMU, arXiv [2410.08146](https://arxiv.org/abs/2410.08146) (ICLR 2025).

What the paper asks for, and what it means here:

| the paper | here |
|---|---|
| a step's process reward is its **progress** -- its advantage, the change in the chance of ending right -- not the value of the state it reaches | a turn's progress `A^μ = f(q after) − f(q before)`, `q` the critic's estimate of the base policy's chance of success |
| progress measured under a **prover** μ distinct from the base policy π; good provers are *complementary*: their advantages tell π's steps apart (distinguishability) and agree with π's own (alignment) -- its Theorem 3.1; too strong or too weak provers give `A^μ ≈ 0` | the prover is **BoK(π)**, the paper's class for its RL, with `Q^{BoK(π)} = 1 − (1 − Q^π)^K` (its Eq. 30); K chosen by the theorem's criterion on 07's episodes |
| the verifier that predicts progress (a PAV) is a **separate model** -- every PAV in the paper fine-tuned from Gemma 9B, for 2B, 9B and 27B policies | the critic is a separate pretrained model, frozen (fine-tuning a 2B model inside every PPO step is beyond this machine), read before each turn; two linear heads refitted at every step on the last 25 steps: v (PPO's baseline) and q (the success probability) |
| online RL with the effective reward `Q^π + α A^μ` per step | every turn's advantage gains `α A^μ` -- its own, not a sum to go |

The candidates: **Qwen3-1.7B** (another family: attention in every layer, a 151k vocabulary, against Qwen3.5's hybrid of gated delta-rule and attention layers with 248k; 2x the policy), **Qwen3.5-2B** (the policy's family one size up -- the paper's best prover across models was one size up), **Qwen3-4B** (the paper's verifier-to-policy ratio, 9B for 2B). And, for comparison, 07's critic's own features.

*(Results written when the runs are in.)*

In [ ]:
import collections, importlib.util, json, math, sys
from pathlib import Path

HERE = Path.cwd() if (Path.cwd() / "08_prover_critic.py").exists() else Path("experiments/notebooks/smoke_test35")
# This notebook's own module (07's, copied and changed as its header says); loaded by path because the file name starts
# with a digit. Nothing below needs WaterTAP or a GPU: every number is read from the runs' records.
spec = importlib.util.spec_from_file_location("pc08", HERE / "08_prover_critic.py")
pc = importlib.util.module_from_spec(spec)
sys.modules["pc08"] = pc
spec.loader.exec_module(pc)
RUNS, RUNS07 = pc.RUNS, pc.RUNS07
SEED07, CTRL07, PPO07 = "seed-n200-r100", "seed-n300-r0", "ppo-n200-r100-turn-s0"


def rows_of(path):
    return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]


def jload(path):
    path = Path(path)
    return json.loads(path.read_text()) if path.exists() else None


FIT = jload(pc.CRITIC_FIT)
CHOICE = (FIT or {}).get("choice")
PPO08 = (f"ppo-{CHOICE['critic']}-bo{CHOICE['prover_k']}-a{CHOICE['alpha']}-s0" if CHOICE else None)
print("critic-fit:", "not run yet" if FIT is None else f"{len(FIT['candidates'])} candidates", "| choice:", CHOICE,
      "| 08's PPO run:", PPO08)

## critic-fit: which model reads the episodes best

The data are 07's own: every step of its PPO run from the recovery seed (8 episodes a step, sampled at T=1 on train questions) and, held out by question, the seed's and the control's 120 dev questions sampled once at T=1. Each episode is replayed through the simulators (`replay`: its calls run again, its budget and malformed-call errors given as the rollout gave them -- the replayed scores match the recorded rewards exactly) and read as **text** -- the prompt as the policy saw it, each turn, the tool responses -- since a separate model has its own tokenizer. A turn's state is the text before it; a candidate's features are its residual stream after a layer, at the last token before the turn.

Two heads per candidate layer: **v**, ridge regression on the turn's return to go (PPO's baseline, the job 07's head did), and **q**, L2 logistic regression on the episode's success -- the decision right, with the simulations that prove it run (the reward pays the decision in full only then): the base policy's `Q^π`, the prover's input. Penalties by 4-fold cross-validation, grouped by question. Two tests:

* **online** -- for every second step of 07's run, the heads fitted on the 25 steps before it and tested on its 8 episodes: the critic as PPO fits and uses it (a few hundred turns to fit on, a policy that keeps moving). The baselines: the window's success rate (`base`), and the turn index alone (`clock`: the window's mean by turn, 07's "position clock" for a turn);
* **held out** -- fitted on every train episode, tested on the dev questions.

The choice was fixed before the numbers: the separate candidate whose best layer has the lowest online q log-loss (the cheaper one within 0.005 nats).

In [ ]:
def fmt(x, spec):
    return "-" if x is None else format(x, spec)


if FIT:
    d = FIT["data"]
    print(f"{d['train_episodes']} train episodes (07's steps {d['train_steps'][0]}-{d['train_steps'][1]}), {d['heldout_episodes']} held out, "
          f"{d['states']} turn states; success {d['success_train']:.3f} train, {d['success_heldout']:.3f} held out; replay: "
          f"{d['rewards_off_recorded']} rewards and {d['call_errors_off_recorded']} call errors off the recorded")
    print(f"\n{'candidate':12} {'layer':>5} | {'online: q logloss':>17} {'base':>6} {'clock':>6} {'AUC':>6} {'AUC t0':>6} {'AUC t>0':>7} "
          f"{'v EV':>6} {'clock':>6} | {'held out: q ll':>14} {'AUC':>6} {'v EV':>6} | {'s/episode':>9}")
    for name, c in FIT["candidates"].items():
        for L, r in c["layers"].items():
            on, ho = r["online"], r["heldout"]
            star = "*" if int(L) == c["best"]["layer"] else " "
            print(f"{name:12} {L:>4}{star} | {on['q_logloss']:17.4f} {on['q_logloss_base']:6.4f} {on['q_logloss_clock']:6.4f} "
                  f"{fmt(on['q_auc'], '6.3f')} {fmt(on['q_auc_first_turn'], '6.3f')} {fmt(on['q_auc_later_turns'], '7.3f')} "
                  f"{fmt(on['v_ev'], '+6.3f')} {fmt(on['v_ev_clock'], '+6.3f')} | {ho['q_logloss']:14.4f} {fmt(ho['q_auc'], '6.3f')} "
                  f"{fmt(ho['v_ev'], '+6.3f')} | {c['seconds_per_episode']:9.2f}")
    print("\n* each candidate's best layer (online q log-loss); 'policy' is 07's critic's features (07's seed, its own layers)")

## The prover: which K

The paper's Theorem 3.1 bounds one policy update's improvement below by **distinguishability** -- the variance of `A^μ` over the base policy's actions at a state -- plus **alignment** -- the mean of `A^μ A^π`. For BoK(π) both are functions of `q` alone. With one recorded action per state, an action's deviation from its state's mean is taken to first order, `f_K'(q_t)(q_{t+1} − q_t)` (`q` is a martingale under π), and exactly `y − q` at the last turn; the plain variance over all turns would add the spread *between* states of `A^μ`'s mean and grow without end in K. Remark 3.1's reading: K > 1 helps a base policy that rarely succeeds (`Q = O(1/K)`), and a large K hurts where it already succeeds (`f_K` saturates). K is the maximiser of D + C on the chosen critic's online predictions; **α** scales `A^μ`'s spread to PPO's own advantage's (the paper tuned α on validation runs, finding 0.5-6 all helpful; a run here is a day).

In [ ]:
if CHOICE:
    c = FIT["candidates"][CHOICE["critic"]]["layers"][str(CHOICE["layer"])]
    for test in ("online", "heldout"):
        tab = c["prover"][test]
        print(f"{CHOICE['critic']} layer {CHOICE['layer']}, {test}:")
        print(f"  {'K':>3} {'D':>8} {'C':>8} {'D+C':>8} {'mean A':>8} {'sd A':>7} {'won':>6} {'corr PPO adv':>12} {'alpha':>6}")
        for k, r in tab.items():
            print(f"  {k:>3} {r['D']:8.4f} {r['C']:8.4f} {r['D_plus_C']:8.4f} {r['mean']:+8.4f} {r['sd']:7.4f} {r['saturated']:6.3f} "
                  f"{fmt(r['corr_ppo_adv'], '12.3f')} {fmt(r['alpha_matching_sd'], '6.2f')}")
    print("\nchoice:", CHOICE)
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        for name, cand in FIT["candidates"].items():
            Ls = sorted(cand["layers"], key=int)
            axes[0].plot([int(L) / cand["depth"] for L in Ls], [cand["layers"][L]["online"]["q_logloss"] for L in Ls], "o-", label=name)
        base = next(iter(next(iter(FIT["candidates"].values()))["layers"].values()))["online"]["q_logloss_base"]
        axes[0].axhline(base, color="gray", ls=":", label="window success rate")
        axes[0].set_xlabel("layer / depth"); axes[0].set_ylabel("online q log-loss"); axes[0].legend(fontsize=8)
        tab = c["prover"]["online"]
        ks = [int(k) for k in tab]
        for key, lab in (("D", "distinguishability D"), ("C", "alignment C"), ("D_plus_C", "D + C")):
            axes[1].plot(ks, [tab[k][key] for k in tab], "o-", label=lab)
        axes[1].set_xscale("log", base=2); axes[1].set_xlabel("K of the prover BoK(pi)"); axes[1].legend(fontsize=8)
        fig.tight_layout()
        fig.savefig(HERE / "08_prover_critic_fit.png", dpi=90)
        plt.close(fig)
        print("figure: 08_prover_critic_fit.png")
    except ImportError:
        pass

## PPO with the separate critic

07's configuration, copied: the turn as the unit of credit (λ 0.95 per turn), lr 2e-6, KL 0.01 to the seed, the turn ends and the answer's scaffolding out of the actor's loss, 3,072 policy tokens, greedy evaluations of 60 dev questions every 40 steps, and 07's early stops (the reward gate, the stall gate at actor steps 40 and 80, the no-regression gate, answer collapse). Changed: the critic -- the chosen model, frozen, its heads refitted every step on the last 25 steps' turns -- and every turn's advantage gains `α A^μ` under BoK(π).

The first 25 steps are **07's own** (`--warm-from`): the same seed draws the same questions, and in those steps the policy is not updated (the critic's warm-up), so their episodes, replayed, fill the new critic's window, and 07's step-0 evaluation is this run's. From step 25 on both runs see the same question at every step; the evaluations fall at the same steps.

In [ ]:
def rolling(xs, k=10):
    out = []
    for i in range(len(xs)):
        w = [x for x in xs[max(0, i - k + 1): i + 1] if x is not None]
        out.append(sum(w) / len(w) if w else None)
    return out


def load_run(d):
    if not (d / "metrics.jsonl").exists():
        return None
    return {"m": rows_of(d / "metrics.jsonl"), "ev": rows_of(d / "eval.jsonl") if (d / "eval.jsonl").exists() else [],
            "end": jload(d / "ending.json"), "cfg": jload(d / "config.json")}


def summary_lines(label, run):
    m, ev, end = run["m"], run["ev"], run["end"]
    print(f"{label}: {len(m)} steps" + (f"; ended at step {end['stopped_at']} ({end['reason']}), best evaluation step {end['best_step']} "
                                        f"({end['best_reward']:.4f})" if end else " (running)"))
    act = [r for r in m if not r["critic_warmup"]]
    for name, rows in (("actor steps", act), ("last 25 steps", act[-25:])):
        if rows:
            mean = lambda k: sum(r[k] for r in rows if r.get(k) is not None) / max(1, sum(r.get(k) is not None for r in rows))  # noqa: E731
            print(f"   {name:14} reward {mean('reward_mean'):.3f} decision {mean('decision_correct'):.3f} evidence {mean('evidence'):.3f} "
                  f"| erred {mean('rec_sim_erred'):.3f} recovered {mean('rec_recovered_given_error'):.3f} | kl {mean('kl_mean'):.4f} "
                  f"| critic ev {mean('value_ev'):+.3f} (clock {mean('value_ev_position'):+.3f})")
    for r in ev:
        rc = r.get("recovery", {})
        g = rc.get("recovered_given_error")
        print(f"     eval @{r['step']:>4}: reward {r['reward']:.4f} decision {r['decision_correct']:.3f} evidence {r['evidence']:.3f} "
              f"answers {r['answer_found']:.3f} | erred {rc.get('sim_erred', float('nan')):.3f} recovered {'-' if g is None else f'{g:.3f}'}"
              + ("  (07's)" if r.get("reused") else ""))


R07 = load_run(RUNS07 / PPO07)
R08 = load_run(RUNS / PPO08) if PPO08 else None
for label, run in ((f"07: {PPO07} (the policy's own critic)", R07), (f"08: {PPO08}", R08)):
    if run:
        summary_lines(label, run)
    else:
        print(f"{label}: not run yet")
if R07 and R08:
    # Paired by step: the same question at every step (the same rng), so the per-step differences pair the runs.
    a = {r["step"]: r["reward_mean"] for r in R07["m"] if not r["critic_warmup"]}
    b = {r["step"]: r["reward_mean"] for r in R08["m"] if not r["critic_warmup"]}
    common = sorted(set(a) & set(b))
    if common:
        diffs = [b[s] - a[s] for s in common]
        mu = sum(diffs) / len(diffs)
        sd = (sum((x - mu) ** 2 for x in diffs) / max(1, len(diffs) - 1)) ** 0.5
        print(f"\nsampled reward, 08 - 07 over the {len(common)} actor steps both ran (same questions): {mu:+.4f} "
              f"(95% interval {mu - 1.96 * sd / len(diffs) ** 0.5:+.4f} to {mu + 1.96 * sd / len(diffs) ** 0.5:+.4f})")
    sep = [r for r in R08["m"] if r.get("critic_ev_turn") is not None]
    if sep:
        mean = lambda k: sum(r[k] for r in sep if r.get(k) is not None) / len(sep)  # noqa: E731
        print(f"08's critic, out of sample per step: ev (turn) {mean('critic_ev_turn'):+.3f}, q brier {mean('q_brier'):.3f} "
              f"(success {mean('success_rate'):.3f}), A^mu sd {mean('prog_std'):.3f} x alpha = {mean('prog_alpha_std'):.3f}, "
              f"PPO advantage sd {mean('adv_std'):.3f}")
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        panels = [("reward_mean", "sampled reward"), ("decision_correct", "decision right"), ("evidence", "proving simulations"),
                  ("rec_recovered_given_error", "erred, then a clean simulate call"), ("kl_mean", "KL to the seed, per token"),
                  ("value_ev", "critic: explained variance (per token)")]
        fig, axes = plt.subplots(2, 3, figsize=(16, 7))
        for ax, (k, title) in zip(axes.flat, panels):
            for run, col, lab in ((R07, "tab:gray", "07"), (R08, "tab:blue", "08")):
                steps = [r["step"] for r in run["m"]]
                ys = [r.get(k) for r in run["m"]]
                ax.plot(steps, [y if y is not None else float("nan") for y in rolling(ys)], "-", color=col, label=lab)
            ax.axvline(25, color="k", lw=0.5, ls=":")
            ax.set_title(title, fontsize=10)
        for run, col in ((R07, "tab:gray"), (R08, "tab:red")):
            for r in run["ev"]:
                axes.flat[0].plot(r["step"], r["reward"], "s", color=col)
        axes.flat[0].legend()
        fig.suptitle("07 (gray) and 08 (blue): training samples, rolling 10; squares: greedy dev evaluations (07 gray, 08 red)")
        fig.tight_layout()
        fig.savefig(HERE / "08_prover_critic_curves.png", dpi=90)
        plt.close(fig)
        print("figure: 08_prover_critic_curves.png")
    except ImportError:
        pass

## Held out

The probes 07 ran for its own PPO run, on 08's checkpoints: greedy on the same 120 test questions and all 120 of holdout_shift, the forced probe (greedy, the 144 dev questions with a slip, each started from its slipped turns) and the 120 dev questions sampled once at T=1. 07's seed, control and PPO checkpoints are read from 07's runs.

In [ ]:
import random
splits = {s: pc.load_split(s) for s in ("dev", "test")}
TEST120 = {c["id"] for c in random.Random(0).sample(splits["test"], 120)}   # probe --limit 120's questions


def recov_line(o):
    r = o["recovery"]
    f = lambda x: "  -  " if x is None else f"{x:.3f}"  # noqa: E731
    return (f"n {o['n']:3d} reward {o['reward']:.3f} decision {o['decision_correct']:.3f} evidence {o['evidence']:.3f} "
            f"answers {o['answer_found']:.3f} | erred {r['sim_erred']:.3f} recovered {f(r['recovered_given_error'])} "
            f"repeats/ep {r['repeats_per_episode']:.2f}")


def summary_of(path, ids=None):
    """A probe's measures from its episodes (on the questions in ``ids`` when given)."""
    ep_path = Path(str(path).replace(".json", ".episodes.jsonl"))
    if not ep_path.exists():
        return None
    rows = [r for r in rows_of(ep_path) if ids is None or r["id"] in ids]
    if not rows:
        return None
    sc = [r["score"] for r in rows]
    n = len(rows)
    return {"n": n, "reward": sum(x["total"] for x in sc) / n, "decision_correct": sum(x["decision_correct"] for x in sc) / n,
            "evidence": sum(x["evidence"] for x in sc) / n,
            "answer_found": sum(pc.extract_answer(r["final_text"])[0] is not None for r in rows) / n,
            "recovery": pc.recovery_block([pc.recovery_stats(r["calls"]) for r in rows], sc)}


# The PPO runs' probes live in <run>/probes-step<N>/ (07's run ended with its best evaluation at its last step, 120:
# one set; an 08 run has one per checkpoint, best and last, or one when they are the same step).
rows_ = [(f"07: control {CTRL07}", RUNS07 / CTRL07 / "epoch1"), (f"07: seed {SEED07}", RUNS07 / SEED07 / "epoch1")]
for tag, run_dir, run in (("07", RUNS07, PPO07), ("08", RUNS, PPO08)):
    if run and (run_dir / run).exists():
        for d in sorted((run_dir / run).glob("probes-step*"), key=lambda p: int(p.name.split("step")[1])):
            rows_.append((f"{tag}: PPO {d.name.replace('probes-', '')}", d))
for split, ids, tag in (("test", TEST120, "test-T0-n120-stop"), ("holdout_shift", None, "holdout_shift-T0-stop"),
                        ("dev, forced (greedy)", None, "dev-T0-stop-forced"), ("dev, 120 at T=1", None, "dev-T1-n120-stop")):
    print(f"{split}:")
    for label, d in rows_:
        o = summary_of(d / f"{tag}.json", ids)
        print(f"  {label:26} " + (recov_line(o) if o else "(not run yet)"))

## Summary

*(Written when the runs are in. Set before them, the three readings:)*

1. **The critic.** Does another model read 07's episodes better than the policy's own features -- online q log-loss and AUC against the window's success rate and the turn clock, v's explained variance against the clock -- and, in PPO, does its value beat the position clock where 07's critic's did not by much?
2. **PPO.** From the same seed, on the same questions at every step: does 08's run beat 07's -- the greedy dev evaluations at steps 40, 80 and 120 (07: 0.586 → 0.658 → 0.648 → ...) and the sampled reward paired by step -- and does the prover's progress move the recovery measures (erred, then a clean simulate call)?
3. **Held out.** 08's best checkpoint against 07's on the 120 test questions and holdout_shift, greedy.